# Wind Forecast Analysis

This notebook analyzes the accuracy of wind power forecasts using historical data.

We compare what was predicted with what actually happened, check how big the errors are, see how forecast timing affects accuracy, and try to understand how reliable wind power really is.

In [1]:
import pandas as pd
import requests

In [2]:
url = "https://wind-forecast-app.onrender.com/wind-data?start=2026-03-01T00:00&end=2026-03-19T00:26&horizon=24"

response = requests.get(url)
data = response.json()

df = pd.DataFrame(data)

df.head()

,time,forecast,error,absError,actual
0,2026-03-01T00:00:00.000Z,NaN,NaN,NaN,NaN
1,2026-03-01T00:30:00.000Z,NaN,NaN,NaN,NaN
2,2026-03-01T01:00:00.000Z,NaN,NaN,NaN,NaN
3,2026-03-01T01:30:00.000Z,NaN,NaN,NaN,NaN
4,2026-03-01T02:00:00.000Z,NaN,NaN,NaN,NaN


## Data Overview

The dataset contains time-series data of wind power generation,

- actual - measured wind power generation (MW)
- forecast - predicted wind power generation (MW)
- Each row represents a 30 min interval

Some rows contain missing values, which are removed in the next step.

In [3]:
df = df[(df["forecast"].notna()) & (df["actual"].notna())]
df.head()

,time,forecast,error,absError,actual
864,2026-03-19T00:00:00.000Z,4089.0,186.0,186.0,3903.0


## Data Cleaning

Rows with missing values (NaN) are removed to ensure accurate analysis.

NaN values removed, because error calculations require both actual and forecast values.

In [4]:
df["error"] = df["forecast"] - df["actual"]

## Error Calculation

Error is calculated as the difference between forecasted and actual values.

Absolute error represents how far the forecast deviates from actual generation, regardless of direction.

This helps to measure forecast accuracy.

In [8]:
mean_error = df["absError"].mean()
median_error = df["absError"].median()
p95_error = df["absError"].quantile(0.95)

print("Mean Error:", mean_error)
print("Median Error:", median_error)
print("P95 Error:", p95_error)

Mean Error: 186.0
Median Error: 186.0
P95 Error: 186.0


## Error Metrics

Mean error represents the average between forecast and actual values.

Median error shows the typical error.

P95 error shows worst case scenarios where forecast errors are too high.

The dataset contains limited valid forecast points due to constraints in the BMRS API.

As a result, error metrics such as mean, median, and P95 appear identical, since they are calculated over a very small number of observations.

This highlights a limitation in the available dataset rather than the forecasting model itself.

In [9]:
horizons = [1, 4, 8, 12, 24]

results = []

for h in horizons:
    url = f"https://wind-forecast-app.onrender.com/wind-data?start=2026-03-01T00:00&end=2026-03-19T00:00&horizon={h}"
    
    data = requests.get(url).json()
    temp_df = pd.DataFrame(data).dropna()
    
    temp_df["abs_error"] = (temp_df["forecast"] - temp_df["actual"]).abs()
    
    results.append({
        "horizon": h,
        "mae": temp_df["abs_error"].mean()
    })

pd.DataFrame(results)

,horizon,mae
0,1,186.0
1,4,186.0
2,8,186.0
3,12,186.0
4,24,186.0


## Horizon vs Error

Normally, when we predict something further into the future, the error should increase because it’s harder to be accurate.

But in this data, the error stays almost the same for all forecast horizons.

This is because the BMRS API only gives the latest forecast for each time point, not the older versions of forecasts.

So even if we change the horizon, we’re still looking at the same forecast data.

That’s why we cant actually see the usual pattern of error increasing with horizon in this dataset.

In [10]:
p10 = df["actual"].quantile(0.10)
mean_actual = df["actual"].mean()

print("P10 (reliable generation):", p10)
print("Average generation:", mean_actual)

P10 (reliable generation): 3903.0
Average generation: 3903.0


## Conclusion

The analysis shows that forecast accuracy cannot be fully evaluated across different horizons due to limitations in the BMRS dataset.

Error metrics indicate consistent deviation, but this is influenced by the availability of only the latest forecast values.

Wind generation is highly variable, and while average values are relatively high, they are not reliable for planning.

Based on the P10 value, we estimate that at least 3903 MW of wind power can be expected 90% of the time.

Therefore, wind power should be considered a variable energy source and supplemented with other stable generation methods.